## Transaction detection

### Problem

Problem: We want to detect events that belong to the same transaction across cases.

Goal: Find sets of events that co-exist and have similar counts (occurance) per case (this supports transactions occuring several times within the same case).

Direction: Build a case-event count table, compute pairwise weighted Jaccard similarity between event vectors, then extract threshold-based cliques as transaction sets.

### Proposed solution

- Create a case-event count table (rows = cases, columns = events).
- Exclude optional events if needed.
- Represent each event as a count vector across cases.
- Compute pairwise **weighted Jaccard similarity** between event vectors:

$$
\mathrm{sim}(x,y)=\frac{\sum_k \min(x_k,y_k)}{\sum_k \max(x_k,y_k)}
$$

- Build a graph where events are nodes and an edge exists when similarity is `>= SIM_THRESHOLD`.
- Extract maximal cliques to obtain strict transaction candidates (all pairs in a clique satisfy the threshold).
- Keep cliques with size `>= MIN_SET_SIZE` and rank them using pairwise similarity statistics.

### Log File Selection

In [ ]:
log_file = '../../logs/SepsisCases2020EventLog.xes'
#log_file = '../../logs/Road_Traffic_Fine_Management_Process.xes'





### Configuration

In [14]:
# Events to ignore.
EXCLUDED_EVENTS = []

# Minimum number of events required for a candidate transaction set.
MIN_SET_SIZE = 2

# Similarity cutoff for connecting two events in the graph.
# If similarity(event_i, event_j) >= SIM_THRESHOLD, an edge is added.
SIM_THRESHOLD = 0.90

print(f'Excluded events: {EXCLUDED_EVENTS}')
print(f"Sim Threshold: {SIM_THRESHOLD}")
print(f'Min set size: {MIN_SET_SIZE}')

Excluded events: []
Sim Threshold: 0.9
Min set size: 2


### Setup and Data Loading

In [15]:
import networkx as nx
import numpy as np
import pandas as pd
import pm4py

In [16]:
event_log = pm4py.read_xes(log_file)
print(f'Loaded events: {len(event_log):,}')
display(event_log.head())


parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

Loaded events: 15,214


,InfectionSuspected,org:group,DiagnosticBlood,DisfuncOrg,SIRSCritTachypnea,Hypotensie,SIRSCritHeartRate,Infusion,DiagnosticArtAstrup,concept:name,...,DiagnosticLacticAcid,lifecycle:transition,Diagnose,Hypoxie,DiagnosticUrinarySediment,DiagnosticECG,case:concept:name,Leucocytes,CRP,LacticAcid
0,True,A,True,True,True,True,True,True,True,ER Registration,...,True,complete,A,False,True,True,A,NaN,NaN,NaN
1,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Leucocytes,...,NaN,complete,NaN,NaN,NaN,NaN,A,9.6,NaN,NaN
2,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CRP,...,NaN,complete,NaN,NaN,NaN,NaN,A,NaN,21.0,NaN
3,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LacticAcid,...,NaN,complete,NaN,NaN,NaN,NaN,A,NaN,NaN,2.2
4,NaN,C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ER Triage,...,NaN,complete,NaN,NaN,NaN,NaN,A,NaN,NaN,NaN


In [17]:
# Keep only needed columns and clean nulls.
CASE_ID = 'case:concept:name'
ACTIVITY = 'concept:name'
TIMESTAMP = 'time:timestamp'

event_log = event_log.drop(columns=['lifecycle:transition'], errors='ignore').copy()
event_log[TIMESTAMP] = pd.to_datetime(event_log[TIMESTAMP], utc=True, errors='coerce')
event_log = event_log.dropna(subset=[CASE_ID, ACTIVITY, TIMESTAMP]).copy()
event_log = event_log.sort_values([CASE_ID, TIMESTAMP]).reset_index(drop=True)

print(f"Cases: {event_log[CASE_ID].nunique():,} | Activities: {event_log[ACTIVITY].nunique():,}")
display(event_log[[CASE_ID, ACTIVITY, TIMESTAMP]].head())




Cases: 1,050 | Activities: 16


,case:concept:name,concept:name,time:timestamp
0,A,ER Registration,2014-10-22 09:15:41+00:00
1,A,Leucocytes,2014-10-22 09:27:00+00:00
2,A,CRP,2014-10-22 09:27:00+00:00
3,A,LacticAcid,2014-10-22 09:27:00+00:00
4,A,ER Triage,2014-10-22 09:33:37+00:00


### Build case-event count table and exclude selected events

In [18]:
# Build case-event count table.
count_matrix = (
    event_log.groupby([CASE_ID, ACTIVITY]).size()
    .rename('count').reset_index()
    .pivot(index=CASE_ID, columns=ACTIVITY, values='count')
    .fillna(0).astype(int)
)

count_matrix

concept:name,Admission IC,Admission NC,CRP,ER Registration,ER Sepsis Triage,ER Triage,IV Antibiotics,IV Liquid,LacticAcid,Leucocytes,Release A,Release B,Release C,Release D,Release E,Return ER
case:concept:name,,,,,,,,,,,,,,,,
A,0,1,7,1,1,1,1,1,1,7,1,0,0,0,0,0
AA,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0
AAA,0,1,1,1,1,1,1,1,1,1,1,0,0,0,0,1
AB,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0
ABA,0,1,4,1,1,1,1,1,1,5,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZV,0,1,2,1,1,1,1,1,1,2,1,0,0,0,0,0
ZW,0,2,2,1,1,1,1,1,1,2,1,0,0,0,0,0
ZX,0,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0


In [19]:
# Exclude selected events and print updated table.
use_cols = [c for c in count_matrix.columns if c not in EXCLUDED_EVENTS]
count_matrix = count_matrix[use_cols].copy()

count_matrix

concept:name,Admission IC,Admission NC,CRP,ER Registration,ER Sepsis Triage,ER Triage,IV Antibiotics,IV Liquid,LacticAcid,Leucocytes,Release A,Release B,Release C,Release D,Release E,Return ER
case:concept:name,,,,,,,,,,,,,,,,
A,0,1,7,1,1,1,1,1,1,7,1,0,0,0,0,0
AA,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0
AAA,0,1,1,1,1,1,1,1,1,1,1,0,0,0,0,1
AB,0,0,1,1,1,1,1,1,1,1,0,0,0,0,0,0
ABA,0,1,4,1,1,1,1,1,1,5,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZV,0,1,2,1,1,1,1,1,1,2,1,0,0,0,0,0
ZW,0,2,2,1,1,1,1,1,1,2,1,0,0,0,0,0
ZX,0,1,1,1,1,1,1,1,1,1,1,0,0,0,0,0


### Pairwise similarity transaction sets

1. Build event count vectors.
2. Compute pairwise **weighted Jaccard similarity** between events.
3. Build a threshold graph from similarity values.
4. Extract maximal cliques and keep sets of size >= `MIN_SET_SIZE`.

Strict rule used here: all events in a transaction set must be mutually similar above the threshold.


#### Event count vectors

Represent each event as a count vector over cases.

- Rows: events
- Columns: cases
- Values: number of times event appears in case

In [20]:
# Build event count vectors from the case-event count matrix.
# Each row is one event profile across all cases.
event_vectors = count_matrix.T.astype(float).copy()

print(f"Events: {event_vectors.shape[0]} | Cases: {event_vectors.shape[1]}")
display(event_vectors.head())

Events: 16 | Cases: 1050


case:concept:name,A,AA,AAA,AB,ABA,AC,ACA,AD,ADA,AE,...,ZQ,ZR,ZS,ZT,ZU,ZV,ZW,ZX,ZY,ZZ
concept:name,,,,,,,,,,,,,,,,,,,,,
Admission IC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Admission NC,1.0,0.0,1.0,0.0,1.0,1.0,0.0,2.0,1.0,2.0,...,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,0.0,1.0
CRP,7.0,1.0,1.0,1.0,4.0,2.0,1.0,7.0,8.0,3.0,...,6.0,3.0,2.0,2.0,1.0,2.0,2.0,1.0,1.0,3.0
ER Registration,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
ER Sepsis Triage,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


#### Pairwise weighted Jaccard similarity

Compute similarity between every pair of events using count vectors:

$$
\mathrm{sim}(x,y)=
\frac{\sum_k \min(x_k,y_k)}
     {\sum_k \max(x_k,y_k)}
$$

Interpretation:
- `1.0` means identical count profile across cases.
- `0.0` means no overlap in counts across cases.


##### Small worked example (4 traces, 2 events)

To make the formula concrete, consider two event count vectors across 4 traces:
- Event A: $x = [1,\,0,\,2,\,1]$
- Event B: $y = [1,\,1,\,1,\,0]$

We compare the vectors trace by trace:
- Overlap part (minimum per trace): $\min(x,y) = [1,\,0,\,1,\,0]$
- Union part (maximum per trace): $\max(x,y) = [1,\,1,\,2,\,1]$

Now sum both parts:
- Numerator: $\sum_k \min(x_k,y_k) = 1+0+1+0 = 2$
- Denominator: $\sum_k \max(x_k,y_k) = 1+1+2+1 = 5$

So the weighted Jaccard similarity is:

$$
\mathrm{sim}(x,y)=\frac{2}{5}=0.4
$$

Reasoning: the two events share some behavior (same non-zero counts in part of the traces), but they also differ in multiple traces. The score $0.4$ reflects **partial overlap**: not dissimilar enough to be 0, and far from identical (which would be 1).

In [21]:
# Compute pairwise weighted Jaccard similarity matrix (events x events).
X = event_vectors.to_numpy()
min_sum = np.minimum(X[:, None, :], X[None, :, :]).sum(axis=2)
max_sum = np.maximum(X[:, None, :], X[None, :, :]).sum(axis=2)

# np.divide computes element-wise min_sum / max_sum; using 'where' skips zero denominators,
# and 'out' pre-fills those skipped positions with 1.0 (for identical all-zero vector pairs).
sim_values = np.divide(
    min_sum,
    max_sum,
    out=np.ones_like(min_sum, dtype=float),
    where=max_sum > 0
    )

similarity_matrix = pd.DataFrame(
    sim_values,
    index=event_vectors.index,
    columns=event_vectors.index
    )


print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"Value range: [{similarity_matrix.values.min():.4f}, {similarity_matrix.values.max():.4f}]")

similarity_matrix

Similarity matrix shape: (16, 16)
Value range: [0.0000, 1.0000]


concept:name,Admission IC,Admission NC,CRP,ER Registration,ER Sepsis Triage,ER Triage,IV Antibiotics,IV Liquid,LacticAcid,Leucocytes,Release A,Release B,Release C,Release D,Release E,Return ER
concept:name,,,,,,,,,,,,,,,,
Admission IC,1.000000,0.086120,0.035868,0.104068,0.103122,0.103774,0.119048,0.118252,0.078338,0.034585,0.122507,0.081250,0.007092,0.036765,0.016529,0.129121
Admission NC,0.086120,1.000000,0.346259,0.558659,0.557961,0.557491,0.534047,0.493056,0.450164,0.334795,0.566357,0.042088,0.021151,0.020305,0.005076,0.248731
CRP,0.035868,0.346259,1.000000,0.304690,0.304387,0.304808,0.252299,0.230840,0.436645,0.850459,0.203488,0.017167,0.007664,0.007357,0.001839,0.089795
ER Registration,0.104068,0.558659,0.304690,1.000000,0.999048,0.997151,0.783810,0.717143,0.519324,0.295820,0.639048,0.053333,0.023810,0.022857,0.005714,0.280000
ER Sepsis Triage,0.103122,0.557961,0.304387,0.999048,1.000000,0.996201,0.784557,0.717827,0.518720,0.295528,0.638095,0.053384,0.023832,0.022879,0.005720,0.279048
ER Triage,0.103774,0.557491,0.304808,0.997151,0.996201,1.000000,0.781576,0.715100,0.518385,0.295939,0.637227,0.053181,0.023742,0.022792,0.005698,0.279202
IV Antibiotics,0.119048,0.534047,0.252299,0.783810,0.784557,0.781576,1.000000,0.914945,0.526000,0.243275,0.654485,0.053957,0.025393,0.025424,0.003632,0.307963
IV Liquid,0.118252,0.493056,0.230840,0.717143,0.717827,0.715100,0.914945,1.000000,0.487265,0.222584,0.609040,0.054759,0.026385,0.025066,0.003968,0.302239
LacticAcid,0.078338,0.450164,0.436645,0.519324,0.518720,0.518385,0.526000,0.487265,1.000000,0.421994,0.393999,0.033967,0.015668,0.014986,0.002725,0.185185


#### Build threshold-based transaction sets (graph + cliques)

Build a graph where:
- each event is a node
- an edge exists if pairwise similarity is `>= SIM_THRESHOLD`

Then extract maximal cliques.
Each clique guarantees that every pair inside the set satisfies the threshold.

In [22]:
# Build a boolean adjacency matrix from the similarity threshold.
# True means two events are connected (similar enough).
adjacency = (similarity_matrix >= SIM_THRESHOLD).copy()
adjacency

concept:name,Admission IC,Admission NC,CRP,ER Registration,ER Sepsis Triage,ER Triage,IV Antibiotics,IV Liquid,LacticAcid,Leucocytes,Release A,Release B,Release C,Release D,Release E,Return ER
concept:name,,,,,,,,,,,,,,,,
Admission IC,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
Admission NC,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
CRP,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
ER Registration,False,False,False,True,True,True,False,False,False,False,False,False,False,False,False,False
ER Sepsis Triage,False,False,False,True,True,True,False,False,False,False,False,False,False,False,False,False
ER Triage,False,False,False,True,True,True,False,False,False,False,False,False,False,False,False,False
IV Antibiotics,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False
IV Liquid,False,False,False,False,False,False,True,True,False,False,False,False,False,False,False,False
LacticAcid,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False


In [23]:
# Remove self-links on the diagonal (an event should not form an edge with itself).
np.fill_diagonal(adjacency.values, False)
adjacency

concept:name,Admission IC,Admission NC,CRP,ER Registration,ER Sepsis Triage,ER Triage,IV Antibiotics,IV Liquid,LacticAcid,Leucocytes,Release A,Release B,Release C,Release D,Release E,Return ER
concept:name,,,,,,,,,,,,,,,,
Admission IC,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
Admission NC,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
CRP,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
ER Registration,False,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False
ER Sepsis Triage,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False
ER Triage,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False
IV Antibiotics,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False
IV Liquid,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False
LacticAcid,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [24]:
# Create an undirected graph from the adjacency matrix.
G = nx.from_pandas_adjacency(adjacency.astype(int))

# Find maximal cliques: fully connected groups of events.
# Keep only cliques that meet the minimum set size.
cliques = [sorted(list(c)) for c in nx.find_cliques(G) if len(c) >= MIN_SET_SIZE]

# Convert cliques to a candidate-set table.
candidate_sets_soft = pd.DataFrame({'event_set': cliques})
candidate_sets_soft['set_size'] = candidate_sets_soft['event_set'].map(len)

# For one candidate set, compute quality from the pairwise similarity submatrix:
# 1) take only rows/cols of events in the set
# 2) keep upper-triangle pairs (i < j) to avoid duplicates and diagonal
# 3) summarize with min and mean pairwise similarity
def set_similarity(events):
    sub = similarity_matrix.loc[events, events].values
    pair_vals = sub[np.triu_indices(len(events), k=1)]
    return float(pair_vals.mean())

candidate_sets_soft['similarity'] = candidate_sets_soft['event_set'].apply(set_similarity)

# Sort best candidates first (bigger sets, then higher similarity).
candidate_sets_soft = candidate_sets_soft.sort_values(
    ['set_size', 'similarity'],
    ascending=[False, False]
).reset_index(drop=True)

print(f"Graph nodes (events): {G.number_of_nodes()}")
print(f"Graph edges (similarity >= {SIM_THRESHOLD}): {G.number_of_edges()}")
print(f"Candidate cliques kept (size >= {MIN_SET_SIZE}): {len(candidate_sets_soft)}")
candidate_sets_soft

Graph nodes (events): 16
Graph edges (similarity >= 0.9): 4
Candidate cliques kept (size >= 2): 2


,event_set,set_size,similarity
0,"[ER Registration, ER Sepsis Triage, ER Triage]",3,0.997467
1,"[IV Antibiotics, IV Liquid]",2,0.914945


# Log from previous executions 


Road_Traffic_Fine_Management_Process.xes

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>event_set</th>
      <th>set_size</th>
      <th>similarity</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>[Add penalty, Insert Fine Notification]</td>
      <td>2</td>
      <td>1.000000</td>
    </tr>
    <tr>
      <th>1</th>
      <td>[Insert Date Appeal to Prefecture, Send Appeal...</td>
      <td>2</td>
      <td>0.988777</td>
    </tr>
  </tbody>
</table>
</div>

BPI_Challenge_2017

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>event_set</th>
      <th>set_size</th>
      <th>similarity</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>[A_Accepted, A_Complete, A_Concept, A_Create A...</td>
      <td>4</td>
      <td>0.997667</td>
    </tr>
    <tr>
      <th>1</th>
      <td>[O_Create Offer, O_Created, O_Sent (mail and o...</td>
      <td>3</td>
      <td>0.949017</td>
    </tr>
    <tr>
      <th>2</th>
      <td>[A_Pending, O_Accepted]</td>
      <td>2</td>
      <td>1.000000</td>
    </tr>
  </tbody>
</table>
</div>

BPI_Challenge_2012

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>event_set</th>
      <th>set_size</th>
      <th>similarity</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>[A_ACTIVATED, A_APPROVED, A_REGISTERED, O_ACCE...</td>
      <td>4</td>
      <td>0.999332</td>
    </tr>
    <tr>
      <th>1</th>
      <td>[O_CREATED, O_SELECTED, O_SENT]</td>
      <td>3</td>
      <td>1.000000</td>
    </tr>
    <tr>
      <th>2</th>
      <td>[A_PARTLYSUBMITTED, A_SUBMITTED]</td>
      <td>2</td>
      <td>1.000000</td>
    </tr>
    <tr>
      <th>3</th>
      <td>[A_ACCEPTED, A_FINALIZED]</td>
      <td>2</td>
      <td>0.980833</td>
    </tr>
  </tbody>
</table>
</div>

SepsisCases2020EventLog

<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>event_set</th>
      <th>set_size</th>
      <th>similarity</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>[ER Registration, ER Sepsis Triage, ER Triage]</td>
      <td>3</td>
      <td>0.997467</td>
    </tr>
    <tr>
      <th>1</th>
      <td>[IV Antibiotics, IV Liquid]</td>
      <td>2</td>
      <td>0.914945</td>
    </tr>
  </tbody>
</table>
</div>